In [5]:
"""
compare_qcnn_qiskit_transpiler_print.py

Confronto risorse dei circuiti QCNN usando il transpiler Qiskit/IBM.

Confronta quattro architetture:
    - hur6_SO4
    - hur8
    - hur9_SU4
    - custom_cartan_transfer

per due encoding:
    - amplitude encoding
    - pairwise fragment encoding

Per ogni combinazione:
    1. costruisce il circuito Qiskit
    2. lo transpila con generate_preset_pass_manager(...)
    3. stampa circuito raw, circuito transpiled, count_ops, depth, size, ecc.

Non salva CSV.

NOTE:
- Per usare un vero backend IBM, imposta USE_REAL_IBM_BACKEND = True
  e scegli IBM_BACKEND_NAME.
- Se USE_REAL_IBM_BACKEND = False, il codice prova a usare un fake backend IBM.
- Se il fake backend non è disponibile nella tua versione Qiskit, usa un transpile
  locale verso basis_gates + coupling_map lineare.
"""

import math
import random
from collections import OrderedDict

import numpy as np

from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager


# ============================================================
# Config
# ============================================================

SEED = 42

N_QUBITS = 8
IMG_SIZE = 16
N_CLASSES = 4

# Se vuoi usare il backend IBM vero:
USE_REAL_IBM_BACKEND = False
IBM_BACKEND_NAME = "ibm_torino"   # cambia con un backend disponibile sul tuo account

# Livello di ottimizzazione Qiskit: 0, 1, 2, 3
OPTIMIZATION_LEVEL = 3

# Stampa il circuito testuale. Con amplitude encoding può diventare molto lungo.
PRINT_RAW_CIRCUIT = False
PRINT_TRANSPILED_CIRCUIT = False

# Se usi fallback locale, questa è una basis IBM-like.
# Nota: sui backend IBM reali il target può avere ECR oppure CX a seconda della macchina.
FALLBACK_BASIS_GATES = ["rz", "sx", "x", "ecr", "measure"]

# Connettività fallback lineare 0-1-2-3-4-5-6-7.
FALLBACK_COUPLING_MAP = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7)]

# Hur architectures: pooling Hur + no final classifier by default.
USE_FINAL_CLASSIFIER_FOR_HUR = False

# Custom architecture: transfer pooling + final two-qubit classifier.
USE_FINAL_CLASSIFIER_FOR_CUSTOM = True

PERIODIC_BOUNDARY = False

# Fragment encoding config
PATCH_SIZE = 8
SUBPATCH_SIZE = 2
N_PATCH_GROUPS = 2
FEATURES_PER_PATCH = 16
FEATURES_PER_ENCODING_STEP = 4
N_ENCODING_STEPS = FEATURES_PER_PATCH // FEATURES_PER_ENCODING_STEP


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)


set_seed(SEED)


# ============================================================
# Dummy inputs and parameters
# ============================================================

def zeros(n):
    return np.zeros(n, dtype=float)


def make_dummy_amplitude_input():
    """
    One 16x16 image flattened to 256 amplitudes.
    Normalized because QuantumCircuit.initialize expects a statevector.
    """
    rng = np.random.default_rng(SEED)
    x = rng.random(IMG_SIZE * IMG_SIZE)
    x = x / np.linalg.norm(x)
    return x


def patch_8x8_to_2x2_means_single(patch):
    """
    One 8x8 patch -> sixteen local 2x2 means.

    Input:
        patch shape: (8, 8)

    Output:
        features shape: (16,)
    """
    patch = patch.reshape(4, 2, 4, 2)
    means = patch.mean(axis=(1, 3))
    return means.reshape(-1)


def image_to_four_patch_features_single(img):
    """
    One 16x16 image -> four patch feature vectors.

    Output:
        patches shape: (4, 16)

    Patch order:
        0 = top-left
        1 = top-right
        2 = bottom-left
        3 = bottom-right
    """
    p0 = img[0:8, 0:8]
    p1 = img[0:8, 8:16]
    p2 = img[8:16, 0:8]
    p3 = img[8:16, 8:16]

    f0 = patch_8x8_to_2x2_means_single(p0)
    f1 = patch_8x8_to_2x2_means_single(p1)
    f2 = patch_8x8_to_2x2_means_single(p2)
    f3 = patch_8x8_to_2x2_means_single(p3)

    patches = np.stack([f0, f1, f2, f3], axis=0)

    # Feature values in [0,1] -> angles in [0, pi].
    return np.pi * patches


def make_dummy_fragment_input():
    rng = np.random.default_rng(SEED)
    img = rng.random((IMG_SIZE, IMG_SIZE))
    return image_to_four_patch_features_single(img)


# ============================================================
# Small Qiskit helpers
# ============================================================

def u3_compat(qc: QuantumCircuit, theta, phi, lam, qubit):
    """
    Qiskit moved from u3 to u. This helper works across versions.
    """
    if hasattr(qc, "u"):
        qc.u(theta, phi, lam, qubit)
    else:
        qc.u3(theta, phi, lam, qubit)


def append_measurements(qc: QuantumCircuit, output_qubits):
    """
    Add classical bits and measure only output qubits.
    """
    qc.measure_all()
    return qc


def two_qubit_gate_count(qc: QuantumCircuit):
    """
    Count operations acting on exactly 2 qubits.
    """
    total = 0
    for inst in qc.data:
        op = inst.operation
        if getattr(op, "num_qubits", 0) == 2:
            total += 1
    return total


def depth_2q(qc: QuantumCircuit):
    """
    Qiskit depth restricted to 2-qubit operations.
    Compatible with recent Qiskit versions.
    """
    try:
        return qc.depth(filter_function=lambda inst: inst.operation.num_qubits == 2)
    except TypeError:
        # Older Qiskit may pass tuple-like CircuitInstruction or old tuple.
        return qc.depth(filter_function=lambda inst: inst[0].num_qubits == 2)


def print_resource_report(label, qc: QuantumCircuit):
    """
    Print Qiskit resource summary.
    """
    print(f"\n[{label}]")
    print("num_qubits:", qc.num_qubits)
    print("num_clbits:", qc.num_clbits)
    print("size:", qc.size())
    print("depth:", qc.depth())
    print("depth_2q:", depth_2q(qc))
    print("2q_gate_count:", two_qubit_gate_count(qc))
    print("count_ops:", dict(qc.count_ops()))


# ============================================================
# Hur convolutional ansatz
# ============================================================

def hur_convolution_circuit6(qc: QuantumCircuit, theta, wires):
    """
    Hur circuit 6 = U_SO4.
    6 parameters.
    """
    a, b = wires
    qc.ry(theta[0], a)
    qc.ry(theta[1], b)
    qc.cx(a, b)
    qc.ry(theta[2], a)
    qc.ry(theta[3], b)
    qc.cx(a, b)
    qc.ry(theta[4], a)
    qc.ry(theta[5], b)


def hur_convolution_circuit8(qc: QuantumCircuit, theta, wires):
    """
    Hur circuit 8.
    10 parameters.
    """
    a, b = wires
    qc.rx(theta[0], a)
    qc.rx(theta[1], b)
    qc.rz(theta[2], a)
    qc.rz(theta[3], b)
    qc.rx(theta[4], a)
    qc.rx(theta[5], b)
    qc.cx(a, b)
    qc.rx(theta[6], a)
    qc.rx(theta[7], b)
    qc.rz(theta[8], a)
    qc.rz(theta[9], b)


def hur_convolution_circuit9(qc: QuantumCircuit, theta, wires):
    """
    Hur circuit 9 = U_SU4.
    15 parameters.
    """
    a, b = wires
    u3_compat(qc, theta[0], theta[1], theta[2], a)
    u3_compat(qc, theta[3], theta[4], theta[5], b)
    qc.cx(a, b)
    qc.ry(theta[6], a)
    qc.rz(theta[7], b)
    qc.cx(b, a)
    qc.ry(theta[8], a)
    qc.cx(a, b)
    u3_compat(qc, theta[9], theta[10], theta[11], a)
    u3_compat(qc, theta[12], theta[13], theta[14], b)


# ============================================================
# Custom Cartan convolution + transfer pooling
# ============================================================

def custom_cartan_convolution_block(qc: QuantumCircuit, theta, wires):
    """
    Custom Cartan-inspired convolution:
        local rotations -> RXX/RYY/RZZ -> local rotations

    11 parameters.
    """
    a, b = wires

    qc.rz(theta[0], a)
    qc.ry(theta[1], a)
    qc.rz(theta[2], b)
    qc.ry(theta[3], b)

    qc.rxx(theta[4], a, b)
    qc.ryy(theta[5], a, b)
    qc.rzz(theta[6], a, b)

    qc.ry(theta[7], a)
    qc.rz(theta[8], a)
    qc.ry(theta[9], b)
    qc.rz(theta[10], b)


def transfer_pooling_pair(qc: QuantumCircuit, theta_pool, discard, keep):
    """
    Custom controlled-transfer pooling.
    3 parameters.
    """
    qc.cx(discard, keep)
    qc.cry(theta_pool[0], discard, keep)
    qc.crz(theta_pool[1], discard, keep)
    qc.ry(theta_pool[2], keep)


def transfer_pooling_layer(qc: QuantumCircuit, theta_pool, pool_pairs):
    for discard, keep in pool_pairs:
        transfer_pooling_pair(qc, theta_pool, discard, keep)


# ============================================================
# Hur pooling
# ============================================================

def controlled_rx_on_zero(qc: QuantumCircuit, angle, control, target):
    """
    Controlled-RX activated when control is |0>.
    """
    qc.x(control)
    qc.crx(angle, control, target)
    qc.x(control)


def hur_pool_pair(qc: QuantumCircuit, theta_pool, control, target):
    """
    Hur pooling:
        CRZ(theta[0]) with control |1>
        CRX(theta[1]) with control |0>
    """
    qc.crz(theta_pool[0], control, target)
    controlled_rx_on_zero(qc, theta_pool[1], control, target)


def hur_pool_layer(qc: QuantumCircuit, theta_pool, pool_pairs):
    for control, target in pool_pairs:
        hur_pool_pair(qc, theta_pool, control, target)


# ============================================================
# Common convolution layer driver
# ============================================================

def convolution_layer_on_wires(qc: QuantumCircuit, conv_fn, theta_conv, active_wires, periodic=False):
    """
    Apply same two-qubit convolution block on even and shifted pairs.
    """
    even_pairs = [
        (active_wires[i], active_wires[i + 1])
        for i in range(0, len(active_wires) - 1, 2)
    ]

    shifted_pairs = [
        (active_wires[i], active_wires[i + 1])
        for i in range(1, len(active_wires) - 1, 2)
    ]

    if periodic and len(active_wires) > 2:
        shifted_pairs.append((active_wires[-1], active_wires[0]))

    for pair in even_pairs:
        conv_fn(qc, theta_conv, pair)

    for pair in shifted_pairs:
        conv_fn(qc, theta_conv, pair)


# ============================================================
# Encodings
# ============================================================

def amplitude_encoding(qc: QuantumCircuit, x):
    """
    Qiskit amplitude encoding via initialize.

    This is intentionally left as initialize, so the Qiskit transpiler shows
    the real synthesized cost after decomposing it for the chosen backend.
    """
    qc.initialize(x, list(range(N_QUBITS)))


def encode_patch_on_qubit_pair(qc: QuantumCircuit, patch, theta_enc_group, wires):
    """
    Encode one 8x8 patch feature vector into one qubit pair.

    patch:
        shape (16,)

    theta_enc_group:
        shape (4, 4)
    """
    a, b = wires

    for s in range(N_ENCODING_STEPS):
        base = 4 * s

        x0 = patch[base + 0]
        x1 = patch[base + 1]
        x2 = patch[base + 2]
        x3 = patch[base + 3]

        qc.ry(x0, a)
        qc.rz(x1, a)
        qc.ry(x2, b)
        qc.rz(x3, b)

        qc.cx(a, b)
        qc.ry(theta_enc_group[s, 0], a)
        qc.ry(theta_enc_group[s, 1], b)

        qc.cx(b, a)
        qc.rz(theta_enc_group[s, 2], a)
        qc.rz(theta_enc_group[s, 3], b)


def pairwise_fragment_encoding(qc: QuantumCircuit, patches, theta_enc):
    """
    Four patches -> four qubit pairs.

    patches shape:
        (4, 16)

    theta_enc shape:
        (2, 4, 4)

    Sharing:
        top-left/top-right -> theta_enc[0]
        bottom-left/bottom-right -> theta_enc[1]
    """
    pair_wires = [
        (0, 1),
        (2, 3),
        (4, 5),
        (6, 7),
    ]

    for patch_idx, wires in enumerate(pair_wires):
        group_idx = 0 if patch_idx in [0, 1] else 1
        encode_patch_on_qubit_pair(
            qc,
            patches[patch_idx],
            theta_enc[group_idx],
            wires,
        )


def apply_encoding(qc: QuantumCircuit, encoding_name, x, theta_enc):
    if encoding_name == "amplitude":
        amplitude_encoding(qc, x)
    elif encoding_name == "fragment":
        pairwise_fragment_encoding(qc, x, theta_enc)
    else:
        raise ValueError(f"Unknown encoding: {encoding_name}")


# ============================================================
# QCNN architecture builders
# ============================================================

def apply_qcnn_hur(qc: QuantumCircuit, conv_fn, theta_conv1, theta_pool1,
                   theta_conv2, theta_pool2, theta_final=None):
    """
    Hur-style QCNN:
        conv -> Hur pooling 8->4
        conv -> Hur pooling 4->2
        optional final conv on output wires
    """
    wires_8 = [0, 1, 2, 3, 4, 5, 6, 7]
    wires_4 = [0, 2, 4, 6]
    output_wires = [0, 4]

    convolution_layer_on_wires(qc, conv_fn, theta_conv1, wires_8, periodic=PERIODIC_BOUNDARY)

    hur_pool_layer(
        qc,
        theta_pool1,
        pool_pairs=[
            (1, 0),
            (3, 2),
            (5, 4),
            (7, 6),
        ],
    )

    convolution_layer_on_wires(qc, conv_fn, theta_conv2, wires_4, periodic=PERIODIC_BOUNDARY)

    hur_pool_layer(
        qc,
        theta_pool2,
        pool_pairs=[
            (2, 0),
            (6, 4),
        ],
    )

    if theta_final is not None:
        conv_fn(qc, theta_final, output_wires)

    return output_wires


def apply_qcnn_custom(qc: QuantumCircuit, theta_conv1, theta_pool1,
                      theta_conv2, theta_pool2, theta_final=None):
    """
    Custom QCNN:
        Cartan conv -> transfer pooling 8->4
        Cartan conv -> transfer pooling 4->2
        optional final Cartan classifier
    """
    wires_8 = [0, 1, 2, 3, 4, 5, 6, 7]
    wires_4 = [0, 2, 4, 6]
    output_wires = [0, 4]

    convolution_layer_on_wires(
        qc,
        custom_cartan_convolution_block,
        theta_conv1,
        wires_8,
        periodic=PERIODIC_BOUNDARY,
    )

    transfer_pooling_layer(
        qc,
        theta_pool1,
        pool_pairs=[
            (1, 0),
            (3, 2),
            (5, 4),
            (7, 6),
        ],
    )

    convolution_layer_on_wires(
        qc,
        custom_cartan_convolution_block,
        theta_conv2,
        wires_4,
        periodic=PERIODIC_BOUNDARY,
    )

    transfer_pooling_layer(
        qc,
        theta_pool2,
        pool_pairs=[
            (2, 0),
            (6, 4),
        ],
    )

    if theta_final is not None:
        custom_cartan_convolution_block(qc, theta_final, output_wires)

    return output_wires


def build_qcnn_circuit(ansatz_name, ansatz_cfg, encoding_name):
    """
    Build raw Qiskit QuantumCircuit for one ansatz/encoding combination.
    """
    qc = QuantumCircuit(N_QUBITS)

    if encoding_name == "amplitude":
        x = make_dummy_amplitude_input()
    elif encoding_name == "fragment":
        x = make_dummy_fragment_input()
    else:
        raise ValueError(f"Unknown encoding: {encoding_name}")

    theta_enc = np.zeros((N_PATCH_GROUPS, N_ENCODING_STEPS, 4), dtype=float)

    conv_params = ansatz_cfg["conv_params"]
    pool_params = ansatz_cfg["pool_params"]

    theta_conv1 = zeros(conv_params)
    theta_pool1 = zeros(pool_params)
    theta_conv2 = zeros(conv_params)
    theta_pool2 = zeros(pool_params)
    theta_final = zeros(conv_params)

    apply_encoding(qc, encoding_name, x, theta_enc)

    if ansatz_name == "custom_cartan_transfer":
        output_wires = apply_qcnn_custom(
            qc,
            theta_conv1,
            theta_pool1,
            theta_conv2,
            theta_pool2,
            theta_final if USE_FINAL_CLASSIFIER_FOR_CUSTOM else None,
        )
    else:
        output_wires = apply_qcnn_hur(
            qc,
            ansatz_cfg["conv_fn"],
            theta_conv1,
            theta_pool1,
            theta_conv2,
            theta_pool2,
            theta_final if USE_FINAL_CLASSIFIER_FOR_HUR else None,
        )

    # Misuro tutti i qubit per rendere il circuito direttamente eseguibile.
    # Se vuoi misurare solo output_wires, puoi sostituire con un registro classico dedicato.
    qc.measure_all()

    return qc, output_wires


# ============================================================
# Backend / compiler setup
# ============================================================

def get_ibm_backend():
    """
    Return real IBM backend if requested, otherwise None.
    """
    if not USE_REAL_IBM_BACKEND:
        return None

    try:
        from qiskit_ibm_runtime import QiskitRuntimeService
    except Exception as exc:
        raise RuntimeError(
            "USE_REAL_IBM_BACKEND=True, ma qiskit_ibm_runtime non è importabile. "
            "Installa qiskit-ibm-runtime oppure metti USE_REAL_IBM_BACKEND=False."
        ) from exc

    service = QiskitRuntimeService()
    backend = service.backend(IBM_BACKEND_NAME)
    return backend


def get_fake_backend():
    """
    Try to get a fake IBM backend available in common Qiskit installations.
    Returns None if not available.
    """
    candidates = []

    # Newer package locations
    try:
        from qiskit_ibm_runtime.fake_provider import FakeSherbrooke, FakeTorino
        candidates.extend([FakeSherbrooke, FakeTorino])
    except Exception:
        pass

    try:
        from qiskit.providers.fake_provider import FakeSherbrooke, FakeManilaV2
        candidates.extend([FakeSherbrooke, FakeManilaV2])
    except Exception:
        pass

    for cls in candidates:
        try:
            return cls()
        except Exception:
            continue

    return None


def transpile_with_qiskit_compiler(qc: QuantumCircuit, backend=None):
    """
    Use Qiskit's preset pass manager when backend is available.
    Otherwise fallback to qiskit.transpile with basis_gates and coupling_map.
    """
    if backend is not None:
        pm = generate_preset_pass_manager(
            backend=backend,
            optimization_level=OPTIMIZATION_LEVEL,
        )
        return pm.run(qc)

    # Fallback locale: non è un backend IBM reale, ma usa un gate set IBM-like.
    coupling_map = CouplingMap(FALLBACK_COUPLING_MAP)

    return transpile(
        qc,
        basis_gates=FALLBACK_BASIS_GATES,
        coupling_map=coupling_map,
        optimization_level=OPTIMIZATION_LEVEL,
    )


def print_backend_info(backend):
    """
    Print useful backend info.
    """
    print("\n" + "=" * 120)
    print("BACKEND INFO")
    print("=" * 120)

    if backend is None:
        print("Backend: None")
        print("Using fallback transpile with:")
        print("basis_gates:", FALLBACK_BASIS_GATES)
        print("coupling_map:", FALLBACK_COUPLING_MAP)
        return

    print("backend:", backend)

    try:
        print("backend name:", backend.name)
    except Exception:
        try:
            print("backend name:", backend.name())
        except Exception:
            pass

    try:
        print("basis_gates:", backend.configuration().basis_gates)
    except Exception as exc:
        print("basis_gates: unavailable via backend.configuration()", exc)

    try:
        print("operation_names / supported instructions:", backend.operation_names)
    except Exception:
        pass

    try:
        print("num_qubits:", backend.num_qubits)
    except Exception:
        try:
            print("num_qubits:", backend.configuration().num_qubits)
        except Exception:
            pass

    try:
        print("coupling_map:", backend.coupling_map)
    except Exception:
        try:
            print("coupling_map:", backend.configuration().coupling_map)
        except Exception:
            pass


# ============================================================
# Run comparison
# ============================================================

def run_one_case(ansatz_name, ansatz_cfg, encoding_name, backend):
    title = f"{ansatz_name} / {encoding_name}"

    print("\n" + "=" * 120)
    print("CASE:", title)
    print("=" * 120)

    qc_raw, output_wires = build_qcnn_circuit(ansatz_name, ansatz_cfg, encoding_name)

    print("Output wires used for class probabilities before measure_all:", output_wires)

    print_resource_report("RAW CIRCUIT", qc_raw)

    if PRINT_RAW_CIRCUIT:
        print("\n[RAW CIRCUIT DRAW]")
        print(qc_raw.draw(output="text", fold=180))

    try:
        qc_transpiled = transpile_with_qiskit_compiler(qc_raw, backend=backend)
    except Exception as exc:
        print(f"\n[ERROR] Qiskit transpilation failed for {title}: {type(exc).__name__}: {exc}")
        return

    print_resource_report("TRANSPILED CIRCUIT", qc_transpiled)

    if PRINT_TRANSPILED_CIRCUIT:
        print("\n[TRANSPILED CIRCUIT DRAW]")
        print(qc_transpiled.draw(output="text", fold=180))


def main():
    print("Qiskit QCNN transpiler comparison")
    print("N_QUBITS:", N_QUBITS)
    print("OPTIMIZATION_LEVEL:", OPTIMIZATION_LEVEL)
    print("USE_REAL_IBM_BACKEND:", USE_REAL_IBM_BACKEND)
    print("IBM_BACKEND_NAME:", IBM_BACKEND_NAME)
    print("USE_FINAL_CLASSIFIER_FOR_HUR:", USE_FINAL_CLASSIFIER_FOR_HUR)
    print("USE_FINAL_CLASSIFIER_FOR_CUSTOM:", USE_FINAL_CLASSIFIER_FOR_CUSTOM)

    backend = None

    if USE_REAL_IBM_BACKEND:
        backend = get_ibm_backend()
    else:
        backend = get_fake_backend()

    print_backend_info(backend)

    ansatzes = OrderedDict({
        "hur6_SO4": {
            "conv_fn": hur_convolution_circuit6,
            "conv_params": 6,
            "pool_params": 2,
        },
        "hur8": {
            "conv_fn": hur_convolution_circuit8,
            "conv_params": 10,
            "pool_params": 2,
        },
        "hur9_SU4": {
            "conv_fn": hur_convolution_circuit9,
            "conv_params": 15,
            "pool_params": 2,
        },
        "custom_cartan_transfer": {
            "conv_fn": custom_cartan_convolution_block,
            "conv_params": 11,
            "pool_params": 3,
        },
    })

    encodings = ["amplitude", "fragment"]

    for ansatz_name, ansatz_cfg in ansatzes.items():
        for encoding_name in encodings:
            run_one_case(ansatz_name, ansatz_cfg, encoding_name, backend=backend)


if __name__ == "__main__":
    main()


Qiskit QCNN transpiler comparison
N_QUBITS: 8
OPTIMIZATION_LEVEL: 3
USE_REAL_IBM_BACKEND: False
IBM_BACKEND_NAME: ibm_torino
USE_FINAL_CLASSIFIER_FOR_HUR: False
USE_FINAL_CLASSIFIER_FOR_CUSTOM: True

BACKEND INFO
backend: <qiskit_ibm_runtime.fake_provider.backends.sherbrooke.fake_sherbrooke.FakeSherbrooke object at 0x000002602F1A17D0>
backend name: fake_sherbrooke
basis_gates: ['ecr', 'id', 'rz', 'sx', 'x']
operation_names / supported instructions: ['sx', 'for_loop', 'reset', 'delay', 'ecr', 'x', 'rz', 'measure', 'if_else', 'switch_case', 'id']
num_qubits: 127
coupling_map: [[1, 0], [1, 2], [3, 2], [4, 3], [4, 15], [5, 4], [6, 5], [7, 6], [7, 8], [8, 9], [10, 9], [10, 11], [11, 12], [12, 13], [14, 0], [14, 18], [16, 8], [17, 12], [17, 30], [18, 19], [19, 20], [20, 33], [21, 20], [21, 22], [22, 15], [23, 22], [23, 24], [25, 24], [26, 16], [26, 25], [26, 27], [28, 27], [29, 28], [29, 30], [31, 30], [31, 32], [32, 36], [33, 39], [34, 24], [35, 28], [35, 47], [36, 51], [37, 38], [38, 39], 